# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [1]:
# YOUR CHANGES HERE
import pandas as pd
import numpy as np

#load data
trans = pd.read_csv("twizzleflu-transitions.tsv", sep='\t') #transitions
rews = pd.read_csv("twizzleflu-rewards.tsv", sep='\t') #rewards

#define parameters 
gamma = 0.999
theta = 1e-8

#get states
states = trans['state'].unique()

#initialize value func
V = {s: 0 for s in states}

#filter to do-nothing action only
policy_action = 'do-nothing'

#value iteraiton
while True:
    delta = 0
    for s in states:
        v = V[s]

        #get reward for this state-acton
        r = rews[(rews.state == s) & (rews.action == policy_action)]['reward'].values[0]

        #compute expected value
        next_states = trans[(trans.state == s) & (trans.action == policy_action)]
        expected_val = 0
        for _, row, in next_states.iterrows():
            expected_val += row['probability'] * (r + gamma * V[row['next_state']])
        V[s] = expected_val
        delta = max(delta, abs(v-V[s]))
    if delta < theta:
        break

#convert to expected discomfort
discom = {s: -V[s] for s in states}

#save in dataframe
df1 = pd.DataFrame({
    "state": list(discom.keys()),
    "expected_discomfort": list(discom.values())
})
df1

,state,expected_discomfort
0,exposed-1,3.383900
1,exposed-2,4.234109
2,exposed-3,5.297934
3,symptoms-1,6.629047
4,symptoms-2,4.982831
5,symptoms-3,1.662787
6,recovered,-0.000000


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [2]:
# YOUR CHANGES HERE
#save as tsv
df1.to_csv("do-nothing-discomfort.tsv", sep='\t', index=False)

Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [3]:
# YOUR CHANGES HERE
#get all actions
actions = trans["action"].unique()

#use same params as before
#initialize value function (again)
V = {s: 0 for s in states}

while True:
    delta = 0
    for s in states:
        v = V[s]
        action_values = []
        
        for a in actions:
            #check if action is valid for this state
            next_states = trans[(trans.state == s) & (trans.action == a)]
            if len(next_states) == 0:
                continue
            
            #get reward
            reward_rows = rews[(rews.state == s) & (rews.action == a)]
            if len(reward_rows) == 0:
                r = 0
            else:
                r = reward_rows["reward"].values[0]
            
            expected_value = 0
            for _, row in next_states.iterrows():
                expected_value += row["probability"] * (r + gamma * V[row["next_state"]])
            
            action_values.append(expected_value)
        
        #best action value
        if len(action_values) > 0:
            V[s] = max(action_values)
        
        delta = max(delta, abs(v - V[s]))
    
    if delta < theta:
        break

#compute policy using final V
policy = {}

for s in states:
    best_action = None
    best_value = -float("inf")

    for a in actions:
        next_states = trans[(trans.state == s) & (trans.action == a)]
        if len(next_states) == 0:
            continue
        
        reward_rows = rews[(rews.state == s) & (rews.action == a)]
        if len(reward_rows) == 0:
            r = 0
        else:
            r = reward_rows["reward"].values[0]
        
        expected_value = 0
        for _, row in next_states.iterrows():
            expected_value += row["probability"] * (r + gamma * V[row["next_state"]])
        
        if expected_value > best_value:
            best_value = expected_value
            best_action = a
    
    policy[s] = best_action

    #save in dataframe
    df2 = pd.DataFrame({
        "state": list(policy.keys()),
        "action": list(policy.values())
    })
df2

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,drink-oj
4,symptoms-2,drink-oj
5,symptoms-3,drink-oj
6,recovered,do-nothing


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [4]:
# YOUR CHANGES HERE
#save as tsv
df2.to_csv("minimum-discomfort-actions.tsv", sep='\t', index=False)

Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [7]:
# YOUR CHANGES HERE
#initialize value function; use same params as before
V_opt = {s: 0 for s in states}

while True:
    delta = 0
    for s in states:
        v = V_opt[s]
        
        a = policy[s]  # use optimal action
        
        next_states = trans[(trans.state == s) & 
                                  (trans.action == a)]
        
        reward_rows = rews[(rews.state == s) & 
                              (rews.action == a)]
        
        if len(reward_rows) == 0:
            r = 0
        else:
            r = reward_rows["reward"].values[0]
        
        expected_value = 0
        for _, row in next_states.iterrows():
            expected_value += row["probability"] * (
                r + gamma * V_opt[row["next_state"]]
            )
        
        V_opt[s] = expected_value
        delta = max(delta, abs(v - V_opt[s]))
    
    if delta < theta:
        break

#find expected discomfort
discomf_opt = {s: -V_opt[s] for s in states}

#save in df
df3 = pd.DataFrame({
     "state": list(discomf_opt.keys()),
     "expected_discomfort": list(discomf_opt.values())
})
df3

,state,expected_discomfort
0,exposed-1,0.742546
1,exposed-2,1.486578
2,exposed-3,2.976131
3,symptoms-1,5.958221
4,symptoms-2,4.480576
5,symptoms-3,1.495513
6,recovered,-0.000000


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [8]:
# YOUR CHANGES HERE
#save as tsv
df3.to_csv("minimum-discomfort-values.tsv", sep='\t', index=False)

Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [12]:
trans.columns

Index(['action', 'state', 'next_state', 'probability'], dtype='str')

In [13]:
trans['state'].unique()

<StringArray>
[ 'exposed-1',  'exposed-2',  'exposed-3', 'symptoms-1', 'symptoms-2',
 'symptoms-3',  'recovered']
Length: 7, dtype: str

In [ ]:
# YOUR CHANGES HERE
#ID sick states
sick_states = [s for s in states if "symptoms" in s.lower()] #instructions say being sick must have symptoms
duration_rewards = []
for _, row in trans.iterrows():
    s = row['state']
    a = row['action']
    if s in sick_states:
        reward = -1
    else:
        reward = 0
    duration_rewards.append({
        "state": s,
        "action": a,
        "reward": reward
    })

#save in df
df4 = pd.DataFrame(duration_rewards)
df4.head(7) #if has symptoms, reward will be -1

,state,action,reward
0,exposed-1,do-nothing,0
1,exposed-1,do-nothing,0
2,exposed-2,do-nothing,0
3,exposed-2,do-nothing,0
4,exposed-3,do-nothing,0
5,exposed-3,do-nothing,0
6,symptoms-1,do-nothing,-1


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [18]:
# YOUR CHANGES HERE
#save as tsv
df4.to_csv("twizzleflu-rewards.tsv", sep='\t', index=False)

Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [19]:
# YOUR CHANGES HERE
#initialize value function;use same params as before
V_duration = {s: 0 for s in states}

while True:
    delta = 0
    for s in states:
        v = V_duration[s]
        
        action_values = []
        
        for a in actions:
            next_states = trans[(trans.state == s) & 
                                      (trans.action == a)]
            
            if len(next_states) == 0:
                continue
            
            reward_rows = df4[(df4.state == s) &
                                            (df4.action == a)]
            
            if len(reward_rows) == 0:
                r = 0
            else:
                r = reward_rows["reward"].values[0]
            
            expected_value = 0
            for _, row in next_states.iterrows():
                expected_value += row["probability"] * (
                    r + gamma * V_duration[row["next_state"]]
                )
            
            action_values.append(expected_value)
        
        if len(action_values) > 0:
            V_duration[s] = max(action_values)
        
        delta = max(delta, abs(v - V_duration[s]))
    
    if delta < theta:
        break

#extract optimal duration policy
policy_duration = {}
for s in states:
    best_action = None
    best_value = -float('inf')

    for a in actions:
        next_states = trans[(trans.state == s) & (trans.action == a)]

        if len(next_states) == 0:
            continue

        reward_rows = df4[(df4.state == s) & (df4.action == a)]

        if len(reward_rows) == 0:
            r = 0
        else:
            r = reward_rows['reward'].values[0]
        
        expected_value = 0
        for _, row in next_states.iterrows():
            expected_value += row["probability"] * (
                r + gamma * V_duration[row["next_state"]]
            )
        
        if expected_value > best_value:
            best_value = expected_value
            best_action = a
    
    policy_duration[s] = best_action

#save as df
df5 = pd.DataFrame({\
    "state": list(policy_duration.keys()),
    "action": list(policy_duration.values())
    })
df5

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,do-nothing
4,symptoms-2,do-nothing
5,symptoms-3,do-nothing
6,recovered,do-nothing


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [20]:
# YOUR CHANGES HERE
#save to tsv
df5.to_csv("minimum-duration-actions.tsv", sep='\t', index=False)

Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [22]:
# YOUR CHANGES HERE
#total return = negative expected sick days (calculated in prev. part)
expected_days = {s: -V_duration[s] for s in states}

#save in df
df6 = pd.DataFrame({
    "state": list(expected_days.keys()),
    "expected_sick_days": list(expected_days.values())
})
df6

,state,expected_sick_days
0,exposed-1,1.239222
1,exposed-2,2.480926
2,exposed-3,4.966818
3,symptoms-1,9.943579
4,symptoms-2,6.640088
5,symptoms-3,3.325574
6,recovered,-0.000000


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [23]:
# YOUR CHANGES HERE
#save as tsv
df6.to_csv("minimum-duration-days.tsv", sep='\t', index=False)

Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

In [24]:
# YOUR CHANGES HERE
V_speed_discomfort = {s: 0 for s in states}

while True:
    delta = 0
    for s in states:
        v = V_speed_discomfort[s]
        
        a = policy_duration[s]
        
        next_states = trans[(trans.state == s) &
                                  (trans.action == a)]
        
        reward_rows = rews[(rews.state == s) &
                              (rews.action == a)]
        
        if len(reward_rows) == 0:
            r = 0
        else:
            r = reward_rows["reward"].values[0]
        
        expected_value = 0
        for _, row in next_states.iterrows():
            expected_value += row["probability"] * (
                r + gamma * V_speed_discomfort[row["next_state"]]
            )
        
        V_speed_discomfort[s] = expected_value
        delta = max(delta, abs(v - V_speed_discomfort[s]))
    
    if delta < theta:
        break

#convert to discomfort
speed_discomfort = {s: -V_speed_discomfort[s] for s in states}
minimize_discomfort = {s: -V_opt[s] for s in states}

#save in df
df7 = pd.DataFrame({
    "state": list(states),
    "speed_discomfort": [speed_discomfort[s] for s in states],
    "minimize_discomfort": [minimize_discomfort[s] for s in states]
})
df7

,state,speed_discomfort,minimize_discomfort
0,exposed-1,0.826147,0.742546
1,exposed-2,1.653949,1.486578
2,exposed-3,3.311209,2.976131
3,symptoms-1,6.629047,5.958221
4,symptoms-2,4.982831,4.480576
5,symptoms-3,1.662787,1.495513
6,recovered,-0.000000,-0.000000


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [25]:
# YOUR CHANGES HERE
#save as tsv
df7.to_csv("policy-comparison.tsv", sep='\t', index=False)

Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

None